# nib — Colab evaluation

**This notebook contains no logic.** It clones, installs, mounts Drive, copies two
files to local disk, and calls two scripts. Every decision lives in
`configs/base.yaml` and every line of code lives in the repository, because the
moment logic moves into a notebook cell the run stops being reproducible.

## What this produces

The project's first real numbers, in two steps that must run **in this order**:

1. **What real handwriting scores on the line pack.** `check_metrics.py` measures
   the FID floor between two disjoint halves of real lines, the recogniser's own
   error rate, and the writer embedding's accuracy — then writes them to
   `references/references_cvl_lines_64.json`.
2. **What the generator scores against that.** `evaluate_generator.py` reads that
   file, so the baseline it reports against is one that was measured here rather
   than one retyped from a previous session.

Step 1 is cheap enough to run on a laptop. Step 2 is why this notebook wants a
GPU: generation is about 220 seconds per line on CPU, and minutes on a T4.

**Why not reuse the phase-1 numbers.** FID 33.72 and writer retrieval 66.9% were
measured on *word* crops. A line is five times wider and holds far more paper per
image, so it lands somewhere else entirely in Inception's feature space. Holding
a generated line against a word-level floor compares two different things.

## Before you start

Under `MyDrive/nib/` you need:

- **`cvl_lines_64.lmdb`** (127 MB, 9,142 records). Upload the copy from
  `data/processed/upload/` — **never** the one in `data/processed/`, which LMDB
  reserves at 8 GB and which transfers as 8 GB.
- **`checkpoints/writer_embedder.pt`**. Without it, writer retrieval scores 3.7%
  on real handwriting and cannot tell a styled generator from an unstyled one.

## 1. Clone and install

`torch` is deliberately not in the base dependencies — Colab's build is matched to
its CUDA driver, and installing ours on top can replace a working build with one
that does not match the GPU.

The `models` extra **is** installed, and its `transformers<5` pin is
load-bearing: Emuru's own code defines `_tied_weights_keys` while 5.x looks for
`all_tied_weights_keys`, so 5.x cannot load the model at all. The same gap breaks
TrOCR's tokenizer.

If pip replaces a `transformers` that was already imported, Colab will ask you to
**restart the runtime**. Do it, then carry on from cell 2 — nothing above needs
re-running except the clone, which is a no-op the second time.

**Expect a pip conflict warning about `gradio`.** Installing `transformers<5`
pulls `huggingface-hub` down to the 0.34-1.0 range it needs, and Colab's
preinstalled `gradio` wants something newer. Nothing here imports gradio, and
the packages that matter -- transformers, diffusers, safetensors -- are
consistent with each other. It is a warning about the environment as a whole,
not a failed install; the line above it says `Building editable for nib-synth
... done`. Cell 2 is the check that actually decides.


In [ ]:
REPO_URL = "https://github.com/omritzabari/nib.git"

%cd /content
![ -d nib ] || git clone $REPO_URL nib
%cd /content/nib
!git pull --ff-only
!pip install -q -e ".[dev,track,models]"

## 2. What are we running on

Record this. If Colab's versions differ from the local ones, the same code can
behave differently in the two places — and the `transformers` major version is
the one that has already cost this project a day.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
import transformers

print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, " <- must be 4.x")
assert transformers.__version__.startswith("4."), "5.x cannot load Emuru; see pyproject"

## 3. Mount Drive and copy what the run needs

**The copy is the point.** Reading the pack record-by-record over Drive would
leave the GPU waiting on network round-trips. One sequential copy of one file,
and every read after it is local.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!mkdir -p /content/nib/data/processed /content/nib/checkpoints
!time cp /content/drive/MyDrive/nib/cvl_lines_64.lmdb /content/nib/data/processed/
!cp /content/drive/MyDrive/nib/checkpoints/writer_embedder.pt /content/nib/checkpoints/
!ls -lh /content/nib/data/processed/ /content/nib/checkpoints/

## 4. Is everything here

One command, and it answers rather than reassures.

Two things will read as missing and both are correct. The **word pack**, because
one pack is required and not both, and this session needs lines. The **raw CVL
images**, because the 5 GB of sources are not copied to a VM that only has to
read a 127 MB pack — the pack is derived from them, which is what makes them
optional here. The only consequence is that CER cannot be measured on this
machine, and it has already been measured on one that has them.

In [ ]:
%cd /content/nib
!python scripts/check_data.py

## 5. What real handwriting scores (T15) — optional, skippable

**Already measured on CPU and committed** as
`references/references_cvl_lines_64.json`:

```
FID floor        19.15     two disjoint halves of real lines   (words: 33.72)
writer top-1     85.8%     the embedding on real lines         (words: 66.9%)
writer top-5     94.4%
CER              11.45%    TrOCR's own error rate, over 300 lines
```

Both of the first two moved a long way from their word-level values, which is
exactly why they had to be re-measured: against the old floor a generated set
would have looked almost twice as good as it is, and the retrieval bar is far
higher than it appeared.

**This cell is optional, and on Colab it measures only two of the three.** CER is
read from the *raw* CVL line images, not from the pack, and the 5 GB of sources
are deliberately not copied here. Cell 4 will say `CVL images ... miss` for the
same reason, and that is correct rather than a problem.

Running it is safe -- `references.update` keeps any figure this run did not
measure, and prints which ones it kept, so a partial run cannot delete a real
number. But it re-measures on GPU what CPU already settled, so skip straight to
cell 6 unless something above looked wrong.

If you do run it: **FID(real, same real)** must be about 0. Anything else means
the feature extraction is broken and no other FID here means anything.

In [ ]:
%cd /content/nib
!python scripts/check_metrics.py \
    --pack data/processed/cvl_lines_64.lmdb \
    --samples 300 \
    --cer-lines 300 \
    --device cuda

## 6. Check the harness before spending an hour on it

Under a minute. The `fake` generator draws the target text in a typeface — every
number it produces is meaningless and every shape is right, which is exactly
what a pipeline check needs.

The first full evaluation spent twelve minutes before dying on a code path
nobody had exercised. This is the cheap version of finding that out.

**What must appear**, or something below is not wired up:

- square brackets after every figure — those are the 95% intervals
- a line beginning `analysis` — the raw terms saved for offline re-examination

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator fake --samples 120 --device cuda

## 7. Eruku on 60 lines first

Twelve minutes and about 0.4 compute units, against an hour and two units for
the full run. It exercises exactly the same path — the checkpoint onto the GPU,
the style-prefix guard, generation, all three metrics with their intervals, and
the saved analysis — on a twentieth of the samples.

60 is the smallest that works: FID refuses to report below 50, because the
estimate is too biased there to mean anything.

Every failure this project has hit in Colab appeared in the first two minutes of
a run. None of them needed an hour to find.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py     --generator eruku     --samples 60     --device cuda

## 8. Eruku (the main run)

Emuru's successor, from the same group. It replaces the stopping heuristic that
cost us 8.7% of the last run with a learned end-of-generation token, and it does
not require the style image's transcription.

**About three hours for 300 lines**, measured: 35 seconds a line against Emuru's 11. Classifier-free guidance runs two forward passes per token rather than one, which is what buys the style fidelity and what costs the time. Roughly 5 compute units.

**If Eruku returns its style image inside its output, this stops within seconds**
with a message beginning `Eruku's output starts with its own style image`. That
would not be a small bug: the output would contain a real crop of the writer's
hand, and retrieval, FID and CER would all move the way success moves.

Watch `truncated`. Emuru scored 8.7% there. A learned stop token should be near
zero, and that single figure is the clearest test of whether the new model
delivers what its title claims.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator eruku \
    --samples 300 \
    --device cuda

## 9. Eruku with nobody's transcription — the deployable case

The same run with `style_text` withheld.

This is the situation an actual user is in: they photograph a page, and nothing
tells the system what it says. Emuru could not be run this way at all — it
required the transcription, so the product would have had to read the page with
TrOCR first and inherit TrOCR's 11.45% error rate.

The gap between this run and the one above is the exact value of knowing what
the style page says — and therefore the exact value of handing a new user a
printed passage to copy instead of asking for a page they already have.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator eruku-no-style-text \
    --samples 300 \
    --device cuda

## 10. Sweep Eruku's guidance scale

The first Eruku run scored CER +13.0 against Emuru's +20.4 and writer retrieval
5.3% against 22.1% — better at the text, four times worse at the hand. That is
the shape of a guidance scale weighted toward the wrong thing, and
classifier-free guidance is the one knob Emuru did not have.

**60 samples is enough here, and only here.** Writer retrieval is a proportion,
so it is not biased by sample count the way FID is; a shift large enough to
explain a fall from 22.1% to 5.3% will show at 60. FID from these runs is *not*
comparable to the 300-sample ones — ignore it.

Each run writes to its own directory named for the scale, so they do not
overwrite each other and `analyse_run.py` can compare them afterwards.

About 35 minutes each.

- **1.0 raises retrieval** — the dial is the cause and it was set too high
- **2.0 raises it** — the cause is the dial but I had the direction backwards
- **neither moves** — the cause is elsewhere, and no more hours go this way

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py     --generator eruku --samples 60 --cfg-scale 1.0 --device cuda

!python scripts/evaluate_generator.py     --generator eruku --samples 60 --cfg-scale 2.0 --device cuda

## 11. Compare the runs, without a GPU

Reads what each run saved and reports whether their intervals separate at all.
Two figures whose intervals overlap are not a difference.

In [ ]:
%cd /content/nib
!python scripts/analyse_run.py outputs/eval_eruku_lines_cfg1 outputs/eval_eruku_lines_cfg2
!python scripts/analyse_run.py outputs/eval_eruku_lines

## 12. Emuru, to reproduce the baseline

Only needed if you want the comparison measured under today's code — full-sample
CER and intervals — rather than read off the earlier run. Skip it otherwise; the
earlier figures are in `docs/phase2-first-evaluation.md`.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator emuru \
    --samples 300 \
    --device cuda

## 13. Keep the results

Each run writes to its own directory named after its generator, so they do not
overwrite one another. `analysis.npz` is the one to keep: it holds the terms the
metrics were computed from, which is what any later question needs.

In [ ]:
!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_* /content/drive/MyDrive/nib/results/
!cp -r /content/nib/references /content/drive/MyDrive/nib/results/
!du -sh /content/drive/MyDrive/nib/results/*

# Printed as well as copied: faster to read here than to fetch off Drive.
!for f in /content/nib/outputs/eval_*/results.json; do echo "== $f"; cat "$f"; done

## What to report back

- the `SUMMARY` block from each run, **with its intervals**
- `truncated` and `empty outputs` from each — Eruku's truncation rate against
  Emuru's 8.7% is the sharpest single test of the new model
- anything that failed, with the full error text

The intervals are the point. Two results whose intervals overlap cannot be told
apart, and until this run the project had no way of knowing which of its
differences were real.